# Understanding CUDA Profiling

Importing Torch and checking the version of it

In [1]:
!nvidia-smi

Mon Aug 31 21:01:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch
import time
print("torch version: ",torch.__version__)


torch version:  2.11.0+cu128


Running a simple `torch.square` operation

In [3]:
a=torch.tensor([2,1])
a

tensor([2, 1])

In [4]:
b=torch.square(a)
b

tensor([4, 1])

In [5]:
x=torch.tensor(2.5)

In [6]:
start = time.time()
y=torch.square(x)
end = time.time()
print(f"torch square result: {y}")
print(f"Time taken for torch.square call: {end-start}")

torch square result: 6.25
Time taken for torch.square call: 0.000370025634765625


This time is fundamentally wrong since the CPU time difference counts the kernel launch overhead when we actually want the GPU time difference.

One thing you have to clearly observe is that the CPU time never actually takes into account the actual GPU time -- meaning that the GPU/CUDA Kernel is being run in `async`. So we have to synchronize the CPU time in a way such that the GPU time is taken into consideration...

So we can use multiple ways to assess the GPU time better than using the CPU's time library

## CUDA Events

In [7]:
print(torch.cuda)
print(torch.cuda.Event)

<module 'torch.cuda' from '/usr/local/lib/python3.13/dist-packages/torch/cuda/__init__.py'>
<class 'torch.cuda.streams.Event'>


In [8]:
start_cuda = torch.cuda.Event(enable_timing=True)
end_cuda = torch.cuda.Event(enable_timing=True)

In [9]:
print(start_cuda)
print(end_cuda)

<torch.cuda.Event uninitialized>
<torch.cuda.Event uninitialized>


Look at the above these events are `uninitialized`!

In [10]:
start_cuda.record()
y=torch.square(x)
end_cuda.record()

In [11]:
print(torch.cuda.synchronize)

<function synchronize at 0x7ffa8c174400>


In [12]:
torch.cuda.synchronize() #forces the CPU to wait until all previously queued CUDA work on the GPU has finished.

In [13]:
print(start_cuda)
print(end_cuda)

<torch.cuda.Event 0x32b63060>
<torch.cuda.Event 0x325fa960>


In [14]:
print("Time Elapsed: ", start_cuda.elapsed_time(end_cuda))

Time Elapsed:  0.34303998947143555


## Pytorch Autograd Profiler

In [15]:
help(torch.randn)

Help on built-in function randn in module torch:

randn(...)
    randn(*size, *, generator=None, out=None, dtype=None, layout=torch.strided, device=None, requires_grad=False, pin_memory=False) -> Tensor


    Returns a tensor filled with random numbers from a normal distribution
    with mean `0` and variance `1` (also called the standard normal
    distribution).

    .. math::
        \text{out}_{i} \sim \mathcal{N}(0, 1)

    For complex dtypes, the tensor is i.i.d. sampled from a `complex normal distribution`_ with zero mean and
    unit variance as

    .. math::
        \text{out}_{i} \sim \mathcal{CN}(0, 1)

    This is equivalent to separately sampling the real :math:`(\operatorname{Re})` and imaginary
    :math:`(\operatorname{Im})` part of :math:`\text{out}_i` as

    .. math::
        \operatorname{Re}(\text{out}_{i}) \sim \mathcal{N}(0, \frac{1}{2}),\quad
        \operatorname{Im}(\text{out}_{i}) \sim \mathcal{N}(0, \frac{1}{2})

    The shape of the tensor is defined by th

In [16]:
x = torch.randn((1024,1024),device='cuda')

In [17]:
x

tensor([[-1.1040, -1.1664,  1.1671,  ..., -0.3021,  0.9619,  0.0328],
        [-0.1234, -0.8846,  0.1969,  ..., -2.0017, -0.0056,  0.2027],
        [ 0.6178,  0.9512, -0.8235,  ..., -1.1860, -1.2526, -1.2632],
        ...,
        [-0.5980,  0.8198,  0.7193,  ..., -1.8096, -0.1492,  0.4562],
        [ 0.3103, -0.6583,  0.0811,  ...,  1.4280, -0.0780,  0.7508],
        [-0.6462, -0.1717,  1.1537,  ..., -0.6215,  0.2455, -1.2034]],
       device='cuda:0')

In [18]:
# this is pytorch's autograd profiler. use this to profile your GPU Workloads and get the time taken for each operation
with torch.autograd.profiler.profile(use_device='cuda') as prof:
    y = torch.square(x)

In [19]:
print(prof)

-------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                     Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
          cudaEventRecord         0.03%      10.489us         0.03%      10.489us      10.489us       0.000us         0.00%       0.000us       0.000us             1  
             aten::square         0.11%      32.386us        99.94%      30.545ms      30.545ms      28.000us         0.09%      30.550ms      30.550ms             1  
          cudaEventRecord         0.01%       1.969us         0.01%       1.969us       1.969us       0.000us         0.00%       0.000us       0.000us        

In [20]:
print(prof.key_averages().table(sort_by="cuda_time_total"))

-------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                     Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
             aten::square         0.11%      32.386us        99.94%      30.545ms      30.545ms      28.000us         0.09%      30.550ms      30.550ms             1  
                aten::pow         0.37%     113.935us        99.81%      30.506ms      30.506ms      30.514ms        99.88%      30.522ms      30.522ms             1  
        aten::result_type         0.01%       2.218us         0.01%       2.218us       2.218us       5.000us         0.02%       5.000us       5.000us        

## Pytorch Profiler with Chrome Trace

In [21]:
from torch.profiler import profile, ProfilerActivity

x_cpu = torch.randn((1024,1024))

In [22]:
with profile(activities=[ProfilerActivity.CPU,ProfilerActivity.CUDA], record_shapes=True) as chrprof:
    x_gpu = x_cpu.to("cuda")
    y = torch.square(x_gpu)

chrprof.export_chrome_trace("trace.json")

/usr/local/lib/python3.13/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


Then you can comfortably understand the activities happening on local chrome trace with ease

## NVIDIA Nsight Compute

> Make sure you have installed `ninja` library

In [23]:
!pip install ninja

In [24]:
# this is a way to load inline C++ code into python
from torch.utils.cpp_extension import load_inline

In [25]:
cpp_source = """
#include <string>

std::string hello_world() {
    return "hello world";
}
"""


In [26]:
cpp_source

'\n#include <string>\n\nstd::string hello_world() {\n    return "hello world";\n}\n'

In [27]:
help(load_inline)

Help on function load_inline in module torch.utils.cpp_extension:

load_inline(
    name,
    cpp_sources,
    cuda_sources=None,
    sycl_sources=None,
    functions=None,
    extra_cflags=None,
    extra_cuda_cflags=None,
    extra_sycl_cflags=None,
    extra_ldflags=None,
    extra_include_paths=None,
    build_directory=None,
    verbose=False,
    with_cuda=None,
    with_sycl=None,
    is_python_module=True,
    with_pytorch_error_handling=True,
    keep_intermediates=True,
    use_pch=False,
    no_implicit_headers=False
)
    Load a PyTorch C++ extension just-in-time (JIT) from string sources.

    This function behaves exactly like :func:`load`, but takes its sources as
    strings rather than filenames. These strings are stored to files in the
    build directory, after which the behavior of :func:`load_inline` is
    identical to :func:`load`.

    See `the
    tests <https://github.com/pytorch/pytorch/blob/master/test/test_cpp_extensions_jit.py>`_
    for good examples of u

In [28]:
module = load_inline(
    name = "hello_world", # name of the module
    cpp_sources=cpp_source, # cpp code
    functions=["hello_world"], # functions to be loaded
    verbose=True # verbose output
)


In [30]:
print(module)
print(module.hello_world())

<module 'hello_world' from '/root/.cache/torch_extensions/py313_cu128/hello_world/hello_world.so'>
hello world


Now implementing inline_load with a kernel function (inline)

In [31]:
cpp_source = r"""
#include <torch/extension.h>

torch::Tensor square_cuda(torch::Tensor input);
"""

cuda_source = r"""
#include <torch/extension.h>

__global__ void square_kernel(const float* input, float* output, int n) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < n) {
        float x = input[idx];
        output[idx] = x * x;
    }
}

torch::Tensor square_cuda(torch::Tensor input) {
    input = input.contiguous();

    auto output = torch::empty_like(input);

    int n = input.numel();
    int threads = 256;
    int blocks = (n + threads - 1) / threads;

    square_kernel<<<blocks, threads>>>(
        input.data_ptr<float>(),
        output.data_ptr<float>(),
        n
    );

    return output;
}
"""

module = load_inline(
    name="square_cuda_extension_clean_v2",
    cpp_sources=cpp_source,
    cuda_sources=cuda_source,
    functions=["square_cuda"],
    extra_cuda_cflags=["-O2"],
    verbose=True,
)

In [32]:
x = torch.randn(1024, 1024, device="cuda", dtype=torch.float32)
y = module.square_cuda(x)

print(torch.allclose(y, x * x))
print(y[:5, :5])

True
tensor([[1.3848e+00, 2.0986e+00, 4.8607e-01, 6.3472e-01, 1.8404e-01],
        [6.8335e-01, 3.1935e+00, 6.0211e-01, 6.6930e-01, 8.8329e-01],
        [1.5821e-04, 8.2094e-01, 7.1068e-01, 1.3528e-01, 9.3850e-01],
        [2.0419e-01, 1.7406e+00, 9.9255e-01, 1.5110e+00, 1.9912e-01],
        [1.2432e+00, 2.4141e-01, 7.0314e-01, 7.4482e+00, 3.0024e+00]],
       device='cuda:0')


## Triton

In [33]:
!pip install triton

In [34]:
import triton
import triton.language as tl

In [35]:
help(triton.jit)

Help on function jit in module triton.runtime.jit:

jit(
    fn: 'Optional[T]' = None,
    *,
    version=None,
    repr: 'Optional[Callable]' = None,
    launch_metadata: 'Optional[Callable]' = None,
    do_not_specialize: 'Optional[Iterable[int | str]]' = None,
    do_not_specialize_on_alignment: 'Optional[Iterable[int | str]]' = None,
    debug: 'Optional[bool]' = None,
    noinline: 'Optional[bool]' = None
) -> 'KernelInterface[T]'
    Decorator for JIT-compiling a function using the Triton compiler.

    :note: When a jit'd function is called, arguments are
        implicitly converted to pointers if they have a :code:`.data_ptr()` method
        and a `.dtype` attribute.

    :note: This function will be compiled and run on the GPU. It will only have access to:

           * python primitives,
           * builtins within the triton package,
           * arguments to this function,
           * other jit'd functions

    :param fn: the function to be jit-compiled
    :type fn: Ca

`@triton.jit` tells Triton: “this Python function is not normal Python anymore; treat it as a GPU kernel and compile it.” When you first call the function with a launch grid like `square_kernel[grid](...)`, Triton traces/compiles the function into GPU code through its compiler pipeline, specializing it for compile-time constants like `BLOCK_SIZE`, tensor dtypes, and target GPU architecture. So your Python-looking code using `tl.load`, `tl.arange`, `tl.store`, etc. becomes low-level GPU code such as PTX/CUBIN. The first call may be slower because compilation happens, but later calls reuse the cached compiled kernel.


In [36]:
help(triton.cdiv)

Help on ConstexprFunction in module triton:

cdiv(...)



In [37]:
help(tl.load)

Help on function load in module triton.language.core:

load(
    pointer,
    mask=None,
    other=None,
    boundary_check=(),
    padding_option='',
    cache_modifier='',
    eviction_policy='',
    volatile=False,
    _semantic=None
)
    Return a tensor of data whose values are loaded from memory at location defined by `pointer`:

        (1) If `pointer` is a single element pointer, a scalar is be loaded.  In
            this case:

            - `mask` and `other` must also be scalars,
            - `other` is implicitly typecast to `pointer.dtype.element_ty`, and
            - `boundary_check` and `padding_option` must be empty.

        (2) If `pointer` is an N-dimensional tensor of pointers, an
            N-dimensional tensor is loaded.  In this case:

            - `mask` and `other` are implicitly broadcast to `pointer.shape`,
            - `other` is implicitly typecast to `pointer.dtype.element_ty`, and
            - `boundary_check` and `padding_option` must be empty.



In [ ]:
help(tl.store)

Help on function store in module triton.language.core:

store(pointer, value, mask=None, boundary_check=(), cache_modifier='', eviction_policy='', _semantic=None)
    Store a tensor of data into memory locations defined by `pointer`.

        (1) If `pointer` is a single element pointer, a scalar is stored.  In
            this case:

            - `mask` must also be scalar, and
            - `boundary_check` and `padding_option` must be empty.

        (2) If `pointer` is an N-dimensional tensor of pointers, an
            N-dimensional block is stored.  In this case:

            - `mask` is implicitly broadcast to `pointer.shape`, and
            - `boundary_check` must be empty.

        (3) If `pointer` is a block pointer defined by `make_block_ptr`, a block
            of data is stored.  In this case:

            - `mask` must be None, and
            - `boundary_check` can be specified to control the behavior of out-of-bound access.

    `value` is implicitly broadcast to `po

In [38]:
@triton.jit 
# Triton JIT Function
def square_kernel(x_ptr, y_ptr, n_elements, BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(axis=0) # gives program id to the instance
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE) 
    mask = offsets < n_elements
    x = tl.load(x_ptr + offsets, mask=mask) 
    y=x*x 
    tl.store(y_ptr + offsets, y, mask=mask)

# Python Launcher for Square Kernel
def square_triton(x):
    y = torch.empty_like(x) 
    n_elements = x.numel()
    grid = lambda meta: (
        triton.cdiv(n_elements, meta["BLOCK_SIZE"]),
    )
    square_kernel[grid](
        x, y, n_elements, BLOCK_SIZE=1024
    )
    return y 

In [39]:
x = torch.randn(1024, 1024, device="cuda", dtype=torch.float32)

y_triton = square_triton(x)
y_torch = x * x

print(torch.allclose(y_triton, y_torch))
print(torch.max(torch.abs(y_triton - y_torch)))

True
tensor(0., device='cuda:0')


In [40]:
help(torch.allclose)

Help on built-in function allclose in module torch:

allclose(...)
    allclose(input: Tensor, other: Tensor, rtol: float = 1e-05, atol: float = 1e-08, equal_nan: bool = False) -> bool

    This function checks if :attr:`input` and :attr:`other` satisfy the condition:

    .. math::
        \lvert \text{input}_i - \text{other}_i \rvert \leq \texttt{atol} + \texttt{rtol} \times \lvert \text{other}_i \rvert

    elementwise, for all elements of :attr:`input` and :attr:`other`. The behaviour of this function is analogous to
    `numpy.allclose <https://numpy.org/doc/stable/reference/generated/numpy.allclose.html>`_

    Args:
        input (Tensor): first tensor to compare
        other (Tensor): second tensor to compare
        atol (float, optional): absolute tolerance. Default: 1e-08
        rtol (float, optional): relative tolerance. Default: 1e-05
        equal_nan (bool, optional): if ``True``, then two ``NaN`` s will be considered equal. Default: ``False``

    Example::

        >

In [41]:
def benchmark(fn, x, iters=100):
    # warmup
    for _ in range(10):
        y = fn(x)
    torch.cuda.synchronize()

    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)

    start.record()
    for _ in range(iters):
        y = fn(x)
    end.record()

    torch.cuda.synchronize()

    return start.elapsed_time(end) / iters  # milliseconds

In [42]:
x = torch.randn(10_000_000, device="cuda", dtype=torch.float32)

triton_ms = benchmark(square_triton, x)
torch_ms = benchmark(lambda z: z * z, x)

print(f"Triton square: {triton_ms:.4f} ms")
print(f"PyTorch x*x:   {torch_ms:.4f} ms")

Triton square: 0.3379 ms
PyTorch x*x:   0.3367 ms


You can also debug triton kernels with `interpret=True` on the JIT decorator...

Let's try reading and interpreting the PTX (note that Triton directly generates IRs instead of C++ Code!) so we need to be able to read/interpret PTX Instructions

In [43]:
x = torch.randn(1024, device="cuda")
y = square_triton(x)
torch.cuda.synchronize()

In [44]:
print(x)
print(y)

tensor([-1.4013, -0.4059,  0.2369,  ...,  0.4257, -0.8444, -1.7099],
       device='cuda:0')
tensor([1.9638, 0.1647, 0.0561,  ..., 0.1812, 0.7130, 2.9237], device='cuda:0')


In [45]:
print(x.numel())

1024


In [46]:
x2=torch.randn((1024,1024))
print(x2.numel())

1048576


In [47]:
help(torch.numel)

Help on built-in function numel in module torch:

numel(...)
    numel(input: Tensor) -> int

    Returns the total number of elements in the :attr:`input` tensor.

    Args:
        input (Tensor): the input tensor.

    Example::

        >>> a = torch.randn(1, 2, 3, 4, 5)
        >>> torch.numel(a)
        120
        >>> a = torch.zeros(4,4)
        >>> torch.numel(a)
        16



In [48]:
compiled = square_kernel[(triton.cdiv(x.numel(), 1024),)](
    x, y, x.numel(),
    BLOCK_SIZE=1024
)

After square_triton(x) has already compiled the kernel, this line is just manually launching the same Triton GPU kernel again, without using your Python wrapper.

It means: launch square_kernel with a grid of ceil(num_elements / 1024) Triton programs, pass x as input, y as output, pass the element count, and use the compile-time constant BLOCK_SIZE=1024. Since the kernel was already compiled for this specialization, Triton likely reuses the cached compiled code and only launches it. Also, compiled is a misleading variable name here: this call usually returns None; the real result is written into y on the GPU.

In [49]:
print(compiled.asm["ptx"])

//
// Generated by LLVM NVPTX Back-End
//

.version 8.7
.target sm_75
.address_size 64

	// .globl	square_kernel           // -- Begin function square_kernel
                                        // @square_kernel
.visible .entry square_kernel(
	.param .u64 .ptr .global .align 1 square_kernel_param_0,
	.param .u64 .ptr .global .align 1 square_kernel_param_1,
	.param .u32 square_kernel_param_2,
	.param .u64 .ptr .global .align 1 square_kernel_param_3,
	.param .u64 .ptr .global .align 1 square_kernel_param_4
)
.reqntid 128
{
	.reg .pred 	%p<3>;
	.reg .b32 	%r<25>;
	.reg .b64 	%rd<8>;
	.loc	1 3 0                           // 4268778447.py:3:0
$L__func_begin0:
	.loc	1 3 0                           // 4268778447.py:3:0

// %bb.0:
	ld.param.b64 	%rd5, [square_kernel_param_0];
	ld.param.b64 	%rd6, [square_kernel_param_1];
$L__tmp0:
	.loc	1 4 24                          // 4268778447.py:4:24
	mov.u32 	%r17, %ctaid.x;
	.loc	1 5 20                          // 4268778447.py:5:20
	shl.b32 	%r18,

In [50]:
print(compiled)

## torch.compile()

`torch.compile` takes a normal PyTorch function or model and tries to make it run faster by capturing its operations into a graph, optimizing that graph, and generating more efficient backend code. Instead of executing every PyTorch op one by one through Python, it can fuse operations, reduce overhead, and choose optimized kernels through backends like TorchInductor. The first call may be slower because compilation happens, but later calls can be faster because the optimized version is reused.


In [51]:
# OG defintion
def square_fn(a):
    a=torch.square(a)
    return a

compiled_sqaure = torch.compile(square_fn) # compiled function

In [52]:
x=torch.randn(1024, device="cuda")

In [53]:
with torch.autograd.profiler.profile(use_device='cuda') as prof_compiled:
    y = compiled_sqaure(x)

In [54]:
print(prof_compiled)

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                        cudaEventRecord         0.00%      23.360us         0.00%      23.360us      23.360us       0.000us         0.00%       0.000us       0.000us             1  
                               TorchDynamo Cache Lookup         0.00%       6.526us         0.00%       6.526us       6.526us      15.000us         0.00%      15.000us      15.000us             1  
         

In [55]:
print(prof_compiled.key_averages().table(sort_by="cuda_time_total"))

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                 dynamo         0.31%       6.717ms        93.63%        2.009s        2.009s       6.748ms         0.31%        2.009s        2.009s             1  
                                   entire_frame_compile         0.01%     148.929us        93.31%        2.002s        2.002s     139.000us         0.01%        2.002s        2.002s             1  
         